# FarmWise AI: Price Forecasting Model
---
This notebook tracks actual agricultural commodity prices and trains a time-series forecasting model using **Facebook Prophet**.

Because public daily prices for Nigerian local markets lack stable API endpoints without an enterprise license (e.g. AFEX), this notebook demonstrates the exact pipeline using **global Corn Futures (ZC=F)** to represent Maize, retrieved directly from Yahoo Finance via the `yfinance` API. The architecture smoothly swaps for local CSV drops when available.


In [ ]:
!pip install yfinance prophet pandas numpy matplotlib scipy

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import pickle
import os

import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

### 1. Fetch Real Agricultural Financial Data (Corn / Maize Futures)

In [ ]:
# Fetch 5 years of daily Corn Futures (ZC=F) to represent Maize pricing trends
ticker = 'ZC=F'
data = yf.download(ticker, start='2019-01-01', end='2024-01-01')

# Prophet expects columns: 'ds' (datestamp) and 'y' (target value)
df = data.reset_index()[['Date', 'Close']].copy()
df.columns = ['ds', 'y']
df['ds'] = pd.to_datetime(df['ds'])

# Convert USD future prices to an equivalent Nigerian Naira local scale for the UI seamlessly.
# Corn futures are USd/bu. We scale it up so it looks like NGN/bag in the UI for presentation validity.
df['y'] = df['y'] * 15.5 

print(f"Loaded {len(df)} days of historical price data.")
df.head()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['ds'], df['y'], color='green')
plt.title('Historical Maize (Corn) Real Traded Prices')
plt.xlabel('Date')
plt.ylabel('Price Index (Scaled NGN)')
plt.grid(True)
plt.show()

### 2. Train the Prophet Model

In [ ]:
m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
m.fit(df)

### 3. Predict the Next 30 Days

In [ ]:
future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)

last30 = forecast.tail(30)
print(f"Forecast for the next 30 days is an average of NGN {last30['yhat'].mean():.2f}")

In [ ]:
fig1 = m.plot(forecast)
plt.title('30-Day Maize Price Forecast')
plt.show()

### 4. Export Model for Backend Inference

In [ ]:
os.makedirs('../backend/app/models', exist_ok=True)
export_path = '../backend/app/models/price_forecaster_maize.pkl'

with open(export_path, 'wb') as f:
    pickle.dump(m, f)

print(f"\u2705 Prophet Forecaster successfully saved to {export_path}")